In [ ]:
# NLGCL Kaggle Notebook
# This notebook runs the NLGCL model (Naturally Existing Neighbor Layers Graph Contrastive Learning) on the Kaggle platform.
# It follows the configuration described in the README.md and adapts the execution environment for Kaggle.
#
# The project relies on [RecBole](https://recbole.io/) and [RecBole-GNN](https://github.com/RUCAIBox/RecBole-GNN).

# ## 1. Environment Setup
# Install necessary dependencies including RecBole and PyTorch Geometric libraries.
# On Kaggle with GPU, these libraries should be installed compatible with the CUDA version.

# Install dependencies
# !pip install -q requirements.txt
# !pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

# ## 2. Clone Repository
# Clone the NLGCL repository to access the code and configuration files.
# We also set up the Python path to ensure we can import modules from the repository.

import os
import sys

# Clone the repository if it doesn't exist
if not os.path.exists('NLGCL'):
    !git clone https://github.com/Jinfeng-Xu/NLGCL.git
    print("Repository cloned.")
else:
    print("Repository already exists.")

# Change working directory to the repository root
if os.path.basename(os.getcwd()) != 'NLGCL':
    # If we are not in NLGCL, try to find it
    if os.path.exists('NLGCL'):
        os.chdir('NLGCL')
        print(f"Changed working directory to: {os.getcwd()}")
    else:
        # If cloned above, it should be here. If running locally in the repo, check if we are already inside.
        # Assuming we might be running this notebook from the repo root itself in some cases.
        pass

# Add current directory to sys.path to allow imports
sys.path.append(os.getcwd())

# ## 3. Dataset Preparation
# The model requires datasets like `Yelp`, `Pinterest`, `QB-Video`, or `Alibaba`.
# RecBole can automatically download these datasets if they are not present.
# We will use RecBole's `create_dataset` to attempt an automatic download.

from recbole.config import Config
from recbole.data import create_dataset as create_recbole_dataset
import os

# Define the dataset name (RecBole uses Capitalized names for download)
dataset_name = 'Yelp' 
# Other options: 'Pinterest', 'QB-Video', 'Alibaba'

# Check if dataset exists, if not, try to download using RecBole
data_path = 'dataset'
if not os.path.exists(os.path.join(data_path, dataset_name)):
    print(f"Dataset '{dataset_name}' not found in '{data_path}'. Attempting to download via RecBole...")
    try:
        # Create a config to trigger download
        # We specify dataset_path to ensure it downloads to the correct folder
        config = Config(model='NLGCL', dataset=dataset_name, config_dict={'data_path': 'dataset/'})
        
        # This step automatically checks and downloads the dataset
        create_recbole_dataset(config)
        print(f"Dataset '{dataset_name}' downloaded/verified successfully.")
        
    except Exception as e:
        print(f"Automatic download failed: {e}")
        print("Please ensure the dataset is uploaded manually or internet access is enabled.")
else:
    print(f"Dataset '{dataset_name}' found.")

# ## 4. Run Training
# We define the configuration arguments similar to `main.py` and run the training process.
# We invoke `run_single_model` directly from python code to see the output in the notebook.

from logging import getLogger
from recbole.utils import init_logger, init_seed, set_color
from recbole_gnn.config import Config
from recbole_gnn.utils import create_dataset, data_preparation, get_model, get_trainer
import argparse
import torch

def run_single_model(args):
    # configurations initialization
    config = Config(
        model=args.model,
        dataset=args.dataset,
        config_file_list=args.config_file_list
    )
    # Ensure data_path matches where we downloaded (default is 'dataset/')
    if 'data_path' not in config:
        config['data_path'] = 'dataset/'

    try:
        assert config["enable_sparse"] in [True, False, None]
    except AssertionError:
        raise ValueError("Your config `enable_sparse` must be `True` or `False` or `None`")
    init_seed(config['seed'], config['reproducibility'])

    # logger initialization
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    # dataset filtering
    dataset = create_dataset(config)
    logger.info(dataset)

    # dataset splitting
    train_data, valid_data, test_data = data_preparation(config, dataset)

    # model loading and initialization
    model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
    logger.info(model)

    # trainer loading and initialization
    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

    # model training
    best_valid_score, best_valid_result = trainer.fit(
        train_data, valid_data, saved=True, show_progress=config['show_progress']
    )

    # model evaluation
    test_result = trainer.evaluate(test_data, load_best_model=True, show_progress=config['show_progress'])

    logger.info(set_color('best valid ', 'yellow') + f': {best_valid_result}')
    logger.info(set_color('test result', 'yellow') + f': {test_result}')


# Define arguments class to mimic argparse
class Args:
    def __init__(self):
        self.model = 'NLGCL'
        self.dataset = 'Yelp' # Use Capitalized name to match downloaded folder
        self.config_files = ''
        self.config_file_list = ['properties/overall.yaml']

# Instantiate arguments
args = Args()

# Run the model
# Ensure GPU is available for faster training
if torch.cuda.is_available():
    print(f"Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU. Training might be slow.")

try:
    if os.path.exists(f'dataset/{args.dataset}'):
         run_single_model(args)
    else:
         print(f"Skipping run because dataset/{args.dataset} is missing.")
         print("Please upload the dataset to run the model.")
except Exception as e:
    print(f"An error occurred: {e}")
    # print full traceback
    import traceback
    traceback.print_exc()